# Native Spam

Optional technical review notebook. The production CLI workflow lives in `../scripts/`.


# 01 | Native SPAM data preparation

**Historical crop-area harmonization for the FABLE Calculator**  
*Production notebook · SPAM source preparation · Native 5 arc-minute grid*

## Purpose

This notebook standardizes the observed **SPAM 2000, 2005, 2010 and 2020** global crop-area releases, preserving their native spatial resolution and source crop classifications. It prepares irrigated and rainfed harvested area, physical area and derived harvesting intensity for later mapping to 56 FABLE products.

**This notebook does not perform FABLE crop mapping, interpolate 2015, or apply the later SPAM2000 MIRCA share corrections.** Those steps belong to downstream notebooks.

### Data sources and conventions

- SPAM 2000 v3.0.7, SPAM 2005 v3r2, SPAM 2010 v2r0 and SPAM 2020 v2r2, using the filenames specified in the source manifest below.
- SPAM 2017 is not included: the available processed release in this workflow was sub-Saharan Africa only.
- The grid remains at **5 arc-minutes**; there is no 30′ resampling.
- Harvested and physical areas are separate quantities. Irrigated/rainfed components and published totals are retained where provided.
- **Source caveat:** raw non-numeric or negative crop-area cells are converted to zero by the inherited parser and counted in QC. This rule is not applied to downstream scientific missing observations. Review these counts before release.
- **SPAM2000 OTHE caveat:** the published physical-area total is available for some records even where the irrigated/rainfed physical split is absent. The inherited correction preserves the total and leaves both components missing; it does not invent an allocation.

### Expected outputs

Under `data/processed/SPAM_FABLE_FINAL/01_native_5arcmin/` (or your configured output directory): native 5′ Parquet files for the four observed years, a country-level native SPAM QC export and supporting source/QC tables. Large raw files and intermediate data are excluded from version control.

### Reproducibility status

**Review candidate — not yet a verified clean-start release.** The source notebook contains a later SPAM2000 OTHE repair that assumes a previously generated 2000 Parquet file. The original first-pass production runner must be recovered/integrated and the complete pipeline rerun from source before publication. This version retains the documented algorithms without silently inventing the missing production step.


In [ ]:
# 01 | Configuration and dependencies
from __future__ import annotations

from pathlib import Path
from itertools import zip_longest
import csv
import importlib.util
import os
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Install pinned dependencies from the repository environment before running.
required_packages = {"duckdb": "duckdb", "pycountry": "pycountry"}
missing_packages = [name for name, module in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing packages: " + ", ".join(missing_packages)
        + ". Install the repository requirements/environment first."
    )
import duckdb
import pycountry

# A portable default plus an optional local folder-picker for desktop Jupyter.
# Use SPAM_PROJECT_ROOT or the repository's config rather than editing code.
DEFAULT_PROJECT_ROOT = Path(os.environ.get("SPAM_PROJECT_ROOT", Path.cwd())).expanduser()
USE_FOLDER_PICKER = False   # Optional on a local desktop; leave False for GitHub/CI.

if USE_FOLDER_PICKER:
    try:
        from tkinter import Tk, filedialog
        root = Tk()
        root.withdraw()
        chosen = filedialog.askdirectory(title="Select the project folder containing Data/SPAM")
        root.destroy()
    except Exception as exc:
        raise RuntimeError("Desktop folder selection is unavailable; use SPAM_PROJECT_ROOT") from exc
    if not chosen:
        raise RuntimeError("No project directory selected")
    PROJECT_ROOT = Path(chosen)
else:
    PROJECT_ROOT = DEFAULT_PROJECT_ROOT

# Source files follow the existing project layout: <root>/Data/SPAM/CSV/...
DATA_ROOT = PROJECT_ROOT / "Data"
SPAM_CSV_ROOT = DATA_ROOT / "SPAM" / "CSV"
FINAL_ROOT = DATA_ROOT / "SPAM_FABLE_FINAL"
OUTPUT_DIR = FINAL_ROOT / "01_native_5arcmin"
TEMP_DIR = OUTPUT_DIR / "_temp_sparse"
QC_DIR = OUTPUT_DIR / "qc"

OBSERVED_YEARS = [2000, 2005, 2010, 2020]
GRID_RESOLUTION_DEGREES = 5.0 / 60.0
CHUNK_SIZE = 50_000
TEXT_ENCODINGS = ["utf-8-sig", "utf-8", "cp1252", "latin1"]

if not SPAM_CSV_ROOT.is_dir():
    raise FileNotFoundError(
        f"SPAM source directory not found: {SPAM_CSV_ROOT}\n"
        "Set the SPAM_PROJECT_ROOT environment variable to the folder containing Data/SPAM/CSV."
    )
for folder in (OUTPUT_DIR, TEMP_DIR, QC_DIR):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
print("Source directory:", SPAM_CSV_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Observed years:", OBSERVED_YEARS)


## 02 | Crop taxonomies and parsing helpers

The source crop codes differ across SPAM releases. This section preserves release-specific taxonomies and native grid helpers; it does not harmonize to FABLE products.

In [ ]:
SPAM_CROPS_2000 = [
    "whea", "rice", "maiz", "barl", "mill", "sorg", "pota",
    "swpy", "cass", "banp", "soyb", "bean", "opul", "sugc",
    "sugb", "coff", "cott", "ofib", "grou", "ooil", "othe",
]

SPAM_CROPS_42 = [
    "whea", "rice", "maiz", "barl", "pmil", "smil", "sorg",
    "ocer", "pota", "swpo", "yams", "cass", "orts", "bean",
    "chic", "cowp", "pige", "lent", "opul", "soyb", "grou",
    "cnut", "oilp", "sunf", "rape", "sesa", "ooil", "sugc",
    "sugb", "cott", "ofib", "acof", "rcof", "coco", "teas",
    "toba", "bana", "plnt", "trof", "temf", "vege", "rest",
]

SPAM_CROPS_2020 = [
    "whea", "rice", "maiz", "barl", "mill", "sorg", "ocer",
    "pota", "swpo", "yams", "cass", "orts", "bean", "chic",
    "cowp", "pige", "lent", "opul", "soyb", "grou", "cnut",
    "oilp", "sunf", "rape", "sesa", "ooil", "sugc", "sugb",
    "cott", "ofib", "coff", "coco", "teas", "toba", "bana",
    "plnt", "trof", "temf", "vege", "rest", "pmil", "rcof",
    "citr", "toma", "onio", "rubb",
]

SPAM_CROPS_BY_YEAR = {
    2000: SPAM_CROPS_2000,
    2005: SPAM_CROPS_42,
    2010: SPAM_CROPS_42,
    2020: SPAM_CROPS_2020,
}

def detect_csv_format(path: Path):
    last_error = None
    for encoding in TEXT_ENCODINGS:
        try:
            with path.open("r", encoding=encoding, newline="") as stream:
                sample = stream.read(100_000)
            try:
                sep = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            except csv.Error:
                sep = ","
            return encoding, sep
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeError(f"Could not decode {path}: {last_error}")

def find_col(columns, candidates, required=True):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise KeyError(f"None of these columns were found: {candidates}")
    return None

def split_crop_column(column, year):
    text = str(column).strip().lower()
    expected = set(SPAM_CROPS_BY_YEAR[year])

    if year == 2000:
        if text in expected:
            return text, "total"
        match = re.fullmatch(r"([a-z0-9]+)_([hils])", text)
        if match and match.group(1) in expected:
            return match.group(1), match.group(2)
        return None

    if year in {2005, 2010}:
        if text in expected:
            return text, "total"
        match = re.fullmatch(r"([a-z0-9]+)_([airhls])", text)
        if match and match.group(1) in expected:
            return match.group(1), match.group(2)
        return None

    if year == 2020 and text in expected:
        return text, "total"

    return None

def detect_crop_columns(columns, sample, year):
    rows = []
    for column in columns:
        parsed = split_crop_column(column, year)
        if parsed is None:
            continue
        crop_code, suffix = parsed
        numeric = pd.to_numeric(sample[column], errors="coerce")
        if len(sample) and numeric.notna().mean() < 0.50:
            continue
        rows.append({"column": column, "crop_code": crop_code, "suffix": suffix})
    return rows

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return np.divide(
        numerator,
        denominator,
        out=np.full(len(numerator), np.nan, dtype="float64"),
        where=denominator.to_numpy() != 0,
    )

def normalize_country_name(value):
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def add_native_grid(x, y):
    lon = pd.to_numeric(x, errors="coerce")
    lat = pd.to_numeric(y, errors="coerce")
    bad = (
        lon.isna() | lat.isna()
        | lon.lt(-180) | lon.gt(180)
        | lat.lt(-90) | lat.gt(90)
    )
    if bad.any():
        raise ValueError(f"Invalid coordinates detected in {int(bad.sum())} rows.")

    ncol = int(round(360 / GRID_RESOLUTION_DEGREES))
    nrow = int(round(180 / GRID_RESOLUTION_DEGREES))

    col = np.floor((lon.to_numpy() + 180) / GRID_RESOLUTION_DEGREES + 1e-9).astype("int32")
    row = np.floor((90 - lat.to_numpy()) / GRID_RESOLUTION_DEGREES + 1e-9).astype("int32")
    col = np.clip(col, 0, ncol - 1)
    row = np.clip(row, 0, nrow - 1)

    center_lon = -180 + (col.astype(float) + 0.5) * GRID_RESOLUTION_DEGREES
    center_lat = 90 - (row.astype(float) + 0.5) * GRID_RESOLUTION_DEGREES

    return pd.DataFrame({
        "grid_row_5min": row,
        "grid_col_5min": col,
        "longitude": center_lon,
        "latitude": center_lat,
        "grid_id_5min": [f"G5_{r:04d}_{c:04d}" for r, c in zip(row, col)],
    }, index=getattr(x, "index", None))

def clean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")
    n_missing = int(numeric.isna().sum())
    n_negative = int(numeric.lt(0).sum())
    # This follows the already-audited raw SPAM preprocessing convention.
    numeric = numeric.mask(numeric < 0, 0.0).fillna(0.0)
    return numeric.astype("float64"), n_missing, n_negative

def append_gzip_csv(df, path, header_written):
    if df.empty:
        return header_written
    df.to_csv(
        path,
        mode="a",
        header=not header_written,
        index=False,
        compression="gzip",
    )
    return True


## 03 | Source manifest and schema audit

Each source file must match the documented release, expected crop count and expected schema. Missing files stop execution rather than being replaced with an alternative release.

In [ ]:
manifest_records = [
    {
        "year": 2000, "area_code": "H", "technology": "embedded",
        "country_column": "stat_code", "country_name_column": None,
        "path": SPAM_CSV_ROOT / "2000"
        / "spam2000v3.0.7_global_harvested-area.dbf-csv" / "spam_h.csv",
    },
    {
        "year": 2000, "area_code": "A", "technology": "embedded",
        "country_column": "stat_code", "country_name_column": None,
        "path": SPAM_CSV_ROOT / "2000"
        / "spam2000v3.0.7_global_physical-area.dbf-csv" / "spam_p.csv",
    },
]

split_defs = {
    2005: {
        "H_folder": SPAM_CSV_ROOT / "2005" / "spam2005v3r2_global_harv_area.csv",
        "A_folder": SPAM_CSV_ROOT / "2005" / "spam2005v3r2_global_phys_area.csv",
        "template": "spam2005V3r2_global_{area}_T{tech}.csv",
        "country": "iso3", "country_name": "name_cntr",
    },
    2010: {
        "H_folder": SPAM_CSV_ROOT / "2010" / "spam2010v2r0_global_harv_area.csv",
        "A_folder": SPAM_CSV_ROOT / "2010" / "spam2010v2r0_global_phys_area.csv",
        "template": "spam2010V2r0_global_{area}_T{tech}.csv",
        "country": "iso3", "country_name": "name_cntr",
    },
    2020: {
        "H_folder": SPAM_CSV_ROOT / "2020" / "spam2020V2r2_global_harvested_area",
        "A_folder": SPAM_CSV_ROOT / "2020" / "spam2020V2r2_global_physical_area",
        "template": "spam2020V2r2_global_{area}_T{tech}.csv",
        "country": "FIPS0", "country_name": "ADM0_NAME",
    },
}

for year, definition in split_defs.items():
    for area_code in ["H", "A"]:
        for tech in ["A", "I", "R"]:
            manifest_records.append({
                "year": year,
                "area_code": area_code,
                "technology": tech,
                "country_column": definition["country"],
                "country_name_column": definition["country_name"],
                "path": definition[f"{area_code}_folder"] / definition["template"].format(
                    area=area_code, tech=tech
                ),
            })

SOURCE_MANIFEST = pd.DataFrame(manifest_records)
SOURCE_MANIFEST["exists"] = SOURCE_MANIFEST["path"].map(Path.exists)

missing = SOURCE_MANIFEST.loc[~SOURCE_MANIFEST["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(
        "Required SPAM files are missing:\n" + "\n".join(map(str, missing))
    )

schema_rows = []
for row in SOURCE_MANIFEST.itertuples(index=False):
    path = Path(row.path)
    encoding, sep = detect_csv_format(path)
    sample = pd.read_csv(path, sep=sep, encoding=encoding, nrows=500, low_memory=False)
    crop_entries = detect_crop_columns(sample.columns, sample, int(row.year))
    schema_rows.append({
        "year": int(row.year),
        "area_code": row.area_code,
        "technology": row.technology,
        "country_column": row.country_column,
        "country_name_column": row.country_name_column,
        "path": path,
        "encoding": encoding,
        "separator": sep,
        "columns": sample.columns.tolist(),
        "crop_entries": crop_entries,
        "crop_count": len({x["crop_code"] for x in crop_entries}),
        "crop_column_count": len(crop_entries),
    })

SOURCE_SCHEMA = pd.DataFrame(schema_rows)

# Expected counts: SPAM 2000 has five technology columns per crop in one file.
for row in SOURCE_SCHEMA.itertuples(index=False):
    expected_crops = len(SPAM_CROPS_BY_YEAR[row.year])
    if row.year == 2000:
        if row.crop_column_count != expected_crops * 5:
            raise ValueError(
                f"SPAM 2000 {row.area_code}: expected {expected_crops*5} crop columns, "
                f"found {row.crop_column_count}."
            )
    else:
        if row.crop_count != expected_crops:
            raise ValueError(
                f"SPAM {row.year} {row.area_code} T{row.technology}: "
                f"expected {expected_crops} crops, found {row.crop_count}."
            )

SOURCE_SCHEMA.assign(path=SOURCE_SCHEMA["path"].astype(str)).drop(
    columns=["columns", "crop_entries"]
).to_csv(QC_DIR / "01_source_schema.csv", index=False)

display(
    SOURCE_SCHEMA[
        ["year", "area_code", "technology", "encoding", "crop_count", "crop_column_count", "path"]
    ]
)
print("Source schema validation: PASS")


## 04 | Country identifiers and the SPAM 2005 coordinate lookup

Country names and FIPS codes are normalized to ISO3; the 2005 `alloc_key` lookup supplies source grid coordinates. Overrides are explicit and retained for auditing.

In [ ]:
RAW_COUNTRY_NAME_TO_ISO3_OVERRIDES = {
    "Bahamas": "BHS", "The Bahamas": "BHS",
    "Bolivia": "BOL", "Bolivia (Plurinational State of)": "BOL",
    "Brunei": "BRN", "Brunei Darussalam": "BRN",
    "Cape Verde": "CPV", "Cabo Verde": "CPV",
    "Congo": "COG", "Democratic Republic of the Congo": "COD",
    "Congo, Dem. Rep.": "COD",
    "Cote dIvoire": "CIV", "Cote d'Ivoire": "CIV", "Côte d'Ivoire": "CIV",
    "Czech Republic": "CZE",
    "Eswatini": "SWZ", "Swaziland": "SWZ",
    "Iran": "IRN", "Iran (Islamic Republic of)": "IRN",
    "Kosovo": "XKX",
    "Laos": "LAO", "Lao People's Democratic Republic": "LAO",
    "Lao Peoples Democratic Republic": "LAO",
    "Micronesia (Federated States of)": "FSM",
    "Moldova": "MDA", "Moldova, Republic of": "MDA",
    "North Korea": "PRK", "Democratic People's Republic of Korea": "PRK",
    "Dem Peoples Rep of Korea": "PRK",
    "Palestine": "PSE", "State of Palestine": "PSE",
    "Russia": "RUS", "Russian Federation": "RUS",
    "South Korea": "KOR", "Republic of Korea": "KOR",
    "Syria": "SYR", "Syrian Arab Republic": "SYR",
    "Taiwan": "TWN", "Taiwan, China": "TWN",
    "Tanzania": "TZA", "Tanzania, United Republic of": "TZA",
    "Turkey": "TUR", "Türkiye": "TUR",
    "United Kingdom": "GBR",
    "United States": "USA", "United States of America": "USA",
    "Venezuela": "VEN", "Venezuela (Bolivarian Republic of)": "VEN",
    "Vietnam": "VNM", "Viet Nam": "VNM",
}
COUNTRY_NAME_TO_ISO3_OVERRIDES = {
    normalize_country_name(k): v
    for k, v in RAW_COUNTRY_NAME_TO_ISO3_OVERRIDES.items()
}
FIPS0_TO_ISO3_OVERRIDES = {
    "IV": "CIV", "KN": "PRK", "LA": "LAO", "WZ": "SWZ",
}

def country_name_to_iso3(country_name, fips0):
    raw_code = "" if pd.isna(fips0) else str(fips0).strip().upper()
    if raw_code in FIPS0_TO_ISO3_OVERRIDES:
        return FIPS0_TO_ISO3_OVERRIDES[raw_code]
    if pd.isna(country_name):
        return None
    cleaned = re.sub(r"\s+", " ", str(country_name).strip())
    normalized = normalize_country_name(cleaned)
    if normalized in COUNTRY_NAME_TO_ISO3_OVERRIDES:
        return COUNTRY_NAME_TO_ISO3_OVERRIDES[normalized]
    try:
        return pycountry.countries.lookup(cleaned).alpha_3
    except LookupError:
        try:
            return pycountry.countries.search_fuzzy(cleaned)[0].alpha_3
        except LookupError:
            return None

# ---- 2020 FIPS0 -> ISO3 ------------------------------------------------------
reference = SOURCE_SCHEMA.loc[
    SOURCE_SCHEMA["year"].eq(2020)
    & SOURCE_SCHEMA["area_code"].eq("H")
    & SOURCE_SCHEMA["technology"].eq("A")
].iloc[0]

pairs = []
for chunk in pd.read_csv(
    reference["path"],
    sep=reference["separator"],
    encoding=reference["encoding"],
    usecols=["FIPS0", "ADM0_NAME"],
    chunksize=CHUNK_SIZE,
    low_memory=False,
):
    pairs.append(chunk[["FIPS0", "ADM0_NAME"]].drop_duplicates())

COUNTRY_2020 = pd.concat(pairs, ignore_index=True).drop_duplicates()
COUNTRY_2020["FIPS0"] = COUNTRY_2020["FIPS0"].astype("string").str.strip().str.upper()
COUNTRY_2020["ADM0_NAME"] = COUNTRY_2020["ADM0_NAME"].astype("string").str.strip()
COUNTRY_2020["country_iso3"] = [
    country_name_to_iso3(n, c)
    for n, c in zip(COUNTRY_2020["ADM0_NAME"], COUNTRY_2020["FIPS0"])
]
if COUNTRY_2020["country_iso3"].isna().any():
    display(COUNTRY_2020.loc[COUNTRY_2020["country_iso3"].isna()])
    raise ValueError("Unresolved SPAM 2020 countries remain.")

SPAM2020_FIPS0_TO_ISO3 = dict(zip(COUNTRY_2020["FIPS0"], COUNTRY_2020["country_iso3"]))
COUNTRY_2020.to_csv(QC_DIR / "01_SPAM2020_country_crosswalk.csv", index=False)

def standardize_country_codes(raw_codes, year, raw_names=None):
    codes = raw_codes.astype("string").str.strip().str.upper()
    result = pd.Series(pd.NA, index=codes.index, dtype="string")

    if year in {2005, 2010}:
        result[:] = codes
        return result

    if year == 2020:
        result = codes.map(SPAM2020_FIPS0_TO_ISO3).astype("string")
        if raw_names is not None:
            unresolved = result.isna()
            if unresolved.any():
                result.loc[unresolved] = [
                    country_name_to_iso3(n, c)
                    for n, c in zip(raw_names.loc[unresolved], codes.loc[unresolved])
                ]
        return result

    if year == 2000:
        already_iso = codes.str.fullmatch(r"[A-Z]{3}", na=False)
        result.loc[already_iso] = codes.loc[already_iso]

        fips_like = codes.str.fullmatch(r"[A-Z]{2}\d*", na=False)
        prefixes = codes.loc[fips_like].str[:2]

        historical = {"TO": "TGO"}
        mapped = prefixes.map(historical)
        remaining = mapped.isna()
        mapped.loc[remaining] = prefixes.loc[remaining].map(SPAM2020_FIPS0_TO_ISO3)
        result.loc[fips_like] = mapped
        return result

    raise ValueError(year)

# ---- 2005 alloc_key -> x/y ---------------------------------------------------
coord_candidates = sorted(
    {
        p for p in (DATA_ROOT / "SPAM").rglob("cell5m_allockey_xy.csv")
        if "_FABLE_processing" not in str(p)
        and "SPAM_FABLE_FINAL" not in str(p)
    },
    key=lambda p: (len(p.parts), len(str(p))),
)
if not coord_candidates:
    raise FileNotFoundError(
        "cell5m_allockey_xy.csv was not found. Extract the SPAM 2005 coordinate lookup "
        "from the original Dataverse package before running this notebook."
    )

COORD_2005_PATH = coord_candidates[0]
coord_enc, coord_sep = detect_csv_format(COORD_2005_PATH)
coord_header = pd.read_csv(COORD_2005_PATH, sep=coord_sep, encoding=coord_enc, nrows=5)
alloc_col = find_col(coord_header.columns, ["alloc_key", "allockey"])
x_col = find_col(coord_header.columns, ["x", "lon", "longitude"])
y_col = find_col(coord_header.columns, ["y", "lat", "latitude"])

COORD_2005 = pd.read_csv(
    COORD_2005_PATH,
    sep=coord_sep,
    encoding=coord_enc,
    usecols=[alloc_col, x_col, y_col],
    low_memory=False,
).rename(columns={alloc_col: "alloc_key", x_col: "x", y_col: "y"})
COORD_2005["alloc_key"] = pd.to_numeric(COORD_2005["alloc_key"], errors="coerce").astype("Int64")
COORD_2005 = COORD_2005.dropna(subset=["alloc_key"]).drop_duplicates("alloc_key")

print("2020 ISO3 countries:", COUNTRY_2020["country_iso3"].nunique())
print("2005 coordinate lookup:", COORD_2005_PATH)
print("2005 coordinate rows:", len(COORD_2005))


## 05 | Release-specific source readers

The inherited processing logic treats SPAM2000 embedded system columns, 2005/2010 aligned A/I/R inputs and 2020 sparse nonaligned inputs differently. In 2020, source discrepancies beyond the documented tolerance remain flagged rather than silently forced to reconcile.

In [ ]:
# YEARS
# -----
# 2000:
#   Embedded production-system columns.
#   Irrigated = I
#   Rainfed   = H + L + S
# 2005 / 2010:
#   Separate A / I / R files.
#   Files are row-aligned.
# 2020:
#   Separate sparse A / I / R files.
#   Files are NOT assumed to be row-aligned.
#   Components are parsed independently and merged by:
#       grid_id_5min
#       country_iso3
#       year
#       source_product_code
# 2020 source diagnostics established:
#   - no duplicate native-5′ crop keys in A/I/R
#   - dominant source-balance difference is ~0.1 ha
#   - TA-only rows are exactly 0.1 ha
#   - one-component-missing cases differ from TA by ~0.1 ha
# Therefore:
#   - sparse absence remains zero; do not invent an I/R component
#   - ~0.1-ha TA-vs-(I+R) differences are retained as rounding provenance
#   - TA missing but positive split component(s) present:
#         reconstruct total from I + R
#   - material inconsistencies > 0.11 ha:
#         retain data + explicit REVIEW flag
#         do not silently modify or discard
# This cell replaces all previous versions of:
#   Cell 5 / 5B / 5C / 5D / 5E
AREA_RTOL = 1e-6
AREA_ATOL_HA = 1e-4

# Empirically established from SPAM 2020 source diagnostics.
SPAM2020_ROUNDING_ATOL_HA = 0.11

# COMMON HELPERS
def schema_crop_lookup(schema_row):

    return {
        x["crop_code"]: x["column"]
        for x in schema_row["crop_entries"]
    }

def schema_embedded_lookup(schema_row):

    return {
        (x["crop_code"], x["suffix"]): x["column"]
        for x in schema_row["crop_entries"]
    }

def source_key_columns(schema_row):

    columns = schema_row["columns"]

    use = [
        schema_row["country_column"]
    ]

    if (
        schema_row["country_name_column"]
        and schema_row["country_name_column"] in columns
    ):
        use.append(
            schema_row["country_name_column"]
        )

    # SPAM 2005: alloc_key rather than direct x/y
    if int(schema_row["year"]) == 2005:

        use.append(
            find_col(
                columns,
                [
                    "alloc_key",
                    "allockey",
                ],
            )
        )

    else:

        use.append(
            find_col(
                columns,
                [
                    "x",
                    "lon",
                    "longitude",
                ],
            )
        )

        use.append(
            find_col(
                columns,
                [
                    "y",
                    "lat",
                    "latitude",
                ],
            )
        )

    return list(
        dict.fromkeys(use)
    )

def prepare_keys(
    chunk,
    schema_row,
):

    year = int(
        schema_row["year"]
    )

    country_col = (
        schema_row["country_column"]
    )

    country_name_col = (
        schema_row["country_name_column"]
    )

    raw_names = (
        chunk[country_name_col]
        if (
            country_name_col
            and country_name_col in chunk.columns
        )
        else None
    )

    iso3 = standardize_country_codes(
        chunk[country_col],
        year,
        raw_names,
    )

    invalid = ~iso3.str.fullmatch(
        r"[A-Z]{3}",
        na=False,
    )

    if invalid.any():

        bad = (
            chunk.loc[
                invalid,
                [country_col],
            ]
            .assign(
                country_iso3=
                iso3.loc[
                    invalid
                ].to_numpy()
            )
            .drop_duplicates()
            .head(100)
        )

        display(bad)

        raise ValueError(
            f"Unresolved country codes in SPAM {year}."
        )

    # SPAM 2005
    # alloc_key -> x/y
    if year == 2005:

        a_col = find_col(
            chunk.columns,
            [
                "alloc_key",
                "allockey",
            ],
        )

        alloc = (
            pd.to_numeric(
                chunk[a_col],
                errors="coerce",
            )
            .astype("Int64")
        )

        coords = (
            pd.DataFrame(
                {
                    "alloc_key": alloc
                },
                index=chunk.index,
            )
            .merge(
                COORD_2005,
                on="alloc_key",
                how="left",
                validate="many_to_one",
            )
        )

        missing_coords = (
            coords[
                [
                    "x",
                    "y",
                ]
            ]
            .isna()
            .any(axis=1)
        )

        if missing_coords.any():

            raise ValueError(
                f"SPAM 2005 contains "
                f"{int(missing_coords.sum())} rows "
                "without alloc_key coordinates."
            )

        grid = (
            add_native_grid(
                coords["x"],
                coords["y"],
            )
            .reset_index(
                drop=True
            )
        )

    # SPAM 2000 / 2010 / 2020
    # direct x/y
    else:

        x_col = find_col(
            chunk.columns,
            [
                "x",
                "lon",
                "longitude",
            ],
        )

        y_col = find_col(
            chunk.columns,
            [
                "y",
                "lat",
                "latitude",
            ],
        )

        grid = (
            add_native_grid(
                chunk[x_col],
                chunk[y_col],
            )
            .reset_index(
                drop=True
            )
        )

    # IMPORTANT:
    # clean positional index, independent of original CSV chunk index.

    grid["country_iso3"] = (
        iso3
        .reset_index(
            drop=True
        )
    )

    grid["year"] = year

    return grid

def make_reader(
    schema_row,
):

    usecols = (
        source_key_columns(
            schema_row
        )
        +
        [
            x["column"]
            for x in schema_row["crop_entries"]
        ]
    )

    usecols = list(
        dict.fromkeys(
            usecols
        )
    )

    processing_encoding = (
        "latin1"
        if int(schema_row["year"]) in {
            2005,
            2010,
        }
        else schema_row["encoding"]
    )

    return pd.read_csv(
        schema_row["path"],
        sep=schema_row["separator"],
        encoding=processing_encoding,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )

# SPAM 2000
# EMBEDDED TECHNOLOGY COLUMNS
def process_2000_area(
    area_code,
    out_path,
):

    schema = (
        SOURCE_SCHEMA.loc[
            SOURCE_SCHEMA["year"].eq(2000)
            &
            SOURCE_SCHEMA["area_code"].eq(area_code)
        ]
        .iloc[0]
    )

    lookup = schema_embedded_lookup(
        schema
    )

    required = {
        (
            crop,
            suffix,
        )
        for crop in SPAM_CROPS_2000
        for suffix in [
            "total",
            "h",
            "l",
            "i",
            "s",
        ]
    }

    if not required.issubset(
        set(lookup)
    ):

        raise ValueError(
            f"SPAM 2000 {area_code}: "
            "missing embedded crop columns."
        )

    if out_path.exists():
        out_path.unlink()

    header_written = False

    qc = {
        "year": 2000,
        "area_code": area_code,
        "reader_mode": "2000_EMBEDDED_HLIS",
        "rows_read": 0,
        "missing_numeric_cells_to_zero": 0,
        "negative_cells_to_zero": 0,
        "rows_written": 0,
    }

    for chunk_no, chunk in enumerate(
        make_reader(schema),
        start=1,
    ):

        keys = prepare_keys(
            chunk,
            schema,
        )

        qc["rows_read"] += len(chunk)

        for crop in SPAM_CROPS_2000:

            total, m0, n0 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "total",
                        )
                    ]
                ]
            )

            irrigated, m1, n1 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "i",
                        )
                    ]
                ]
            )

            h, m2, n2 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "h",
                        )
                    ]
                ]
            )

            l, m3, n3 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "l",
                        )
                    ]
                ]
            )

            s, m4, n4 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "s",
                        )
                    ]
                ]
            )

            rainfed = (
                h
                + l
                + s
            )

            computed = (
                irrigated
                + rainfed
            )

            qc[
                "missing_numeric_cells_to_zero"
            ] += (
                m0
                + m1
                + m2
                + m3
                + m4
            )

            qc[
                "negative_cells_to_zero"
            ] += (
                n0
                + n1
                + n2
                + n3
                + n4
            )

            active = (
                total.ne(0)
                |
                irrigated.ne(0)
                |
                rainfed.ne(0)
            )

            if not active.any():
                continue

            # IMPORTANT positional mask fix
            mask = active.to_numpy(
                dtype=bool
            )

            part = (
                keys
                .iloc[mask]
                .copy()
            )

            part[
                "source_product_code"
            ] = crop

            part[
                "source_total_source_ha"
            ] = total.to_numpy()[mask]

            part[
                "source_irrigated_ha"
            ] = irrigated.to_numpy()[mask]

            part[
                "source_rainfed_ha"
            ] = rainfed.to_numpy()[mask]

            part[
                "source_total_ha"
            ] = computed.to_numpy()[mask]

            part[
                "source_balance_residual_ha"
            ] = (
                total.to_numpy()[mask]
                -
                computed.to_numpy()[mask]
            )

            header_written = append_gzip_csv(
                part,
                out_path,
                header_written,
            )

            qc[
                "rows_written"
            ] += len(part)

        print(
            f"SPAM 2000 {area_code}: "
            f"chunk {chunk_no} complete."
        )

    return qc

# SPAM 2005 / 2010
# ROW-ALIGNED A / I / R
def process_aligned_split_area(
    year,
    area_code,
    out_path,
):

    selected = SOURCE_SCHEMA.loc[
        SOURCE_SCHEMA["year"].eq(year)
        &
        SOURCE_SCHEMA["area_code"].eq(area_code)
    ]

    rows = {
        r["technology"]: r
        for _, r in selected.iterrows()
    }

    if set(rows) != {
        "A",
        "I",
        "R",
    }:

        raise ValueError(
            f"SPAM {year} {area_code}: "
            "A/I/R files not complete."
        )

    lookups = {
        tech:
            schema_crop_lookup(row)
        for tech, row in rows.items()
    }

    expected = (
        SPAM_CROPS_BY_YEAR[
            year
        ]
    )

    for tech in [
        "A",
        "I",
        "R",
    ]:

        missing = (
            set(expected)
            -
            set(
                lookups[tech]
            )
        )

        if missing:

            raise ValueError(
                f"SPAM {year} "
                f"{area_code} "
                f"T{tech}: "
                f"missing crops "
                f"{sorted(missing)}"
            )

    if out_path.exists():
        out_path.unlink()

    readers = {
        tech:
            make_reader(
                rows[tech]
            )
        for tech in [
            "A",
            "I",
            "R",
        ]
    }

    header_written = False

    qc = {
        "year": year,
        "area_code": area_code,
        "reader_mode": "ALIGNED_AIR",
        "rows_read": 0,
        "missing_numeric_cells_to_zero": 0,
        "negative_cells_to_zero": 0,
        "rows_written": 0,
    }

    for chunk_no, triple in enumerate(
        zip_longest(
            readers["A"],
            readers["I"],
            readers["R"],
        ),
        start=1,
    ):

        if any(
            x is None
            for x in triple
        ):

            raise ValueError(
                f"SPAM {year} "
                f"{area_code}: "
                "A/I/R files have different row counts."
            )

        chunks = dict(
            zip(
                [
                    "A",
                    "I",
                    "R",
                ],
                triple,
            )
        )

        prepared = {
            tech:
                prepare_keys(
                    chunks[tech],
                    rows[tech],
                )
            for tech in [
                "A",
                "I",
                "R",
            ]
        }

        # Confirm true positional alignment
        base_signature = (
            prepared["A"][
                [
                    "grid_id_5min",
                    "country_iso3",
                ]
            ]
            .reset_index(
                drop=True
            )
        )

        for tech in [
            "I",
            "R",
        ]:

            other = (
                prepared[tech][
                    [
                        "grid_id_5min",
                        "country_iso3",
                    ]
                ]
                .reset_index(
                    drop=True
                )
            )

            if not base_signature.equals(
                other
            ):

                raise ValueError(
                    f"SPAM {year} "
                    f"{area_code}: "
                    "source rows are not aligned "
                    "between A/I/R files "
                    f"in chunk {chunk_no}."
                )

        keys = prepared["A"]

        qc["rows_read"] += len(keys)

        for crop in expected:

            total, m0, n0 = clean_numeric(
                chunks["A"][
                    lookups["A"][crop]
                ]
            )

            irrigated, m1, n1 = clean_numeric(
                chunks["I"][
                    lookups["I"][crop]
                ]
            )

            rainfed, m2, n2 = clean_numeric(
                chunks["R"][
                    lookups["R"][crop]
                ]
            )

            computed = (
                irrigated
                + rainfed
            )

            qc[
                "missing_numeric_cells_to_zero"
            ] += (
                m0
                + m1
                + m2
            )

            qc[
                "negative_cells_to_zero"
            ] += (
                n0
                + n1
                + n2
            )

            active = (
                total.ne(0)
                |
                irrigated.ne(0)
                |
                rainfed.ne(0)
            )

            if not active.any():
                continue

            # IMPORTANT positional mask fix
            mask = active.to_numpy(
                dtype=bool
            )

            part = (
                keys
                .iloc[mask]
                .copy()
            )

            part[
                "source_product_code"
            ] = crop

            part[
                "source_total_source_ha"
            ] = total.to_numpy()[mask]

            part[
                "source_irrigated_ha"
            ] = irrigated.to_numpy()[mask]

            part[
                "source_rainfed_ha"
            ] = rainfed.to_numpy()[mask]

            part[
                "source_total_ha"
            ] = computed.to_numpy()[mask]

            part[
                "source_balance_residual_ha"
            ] = (
                total.to_numpy()[mask]
                -
                computed.to_numpy()[mask]
            )

            header_written = append_gzip_csv(
                part,
                out_path,
                header_written,
            )

            qc[
                "rows_written"
            ] += len(part)

        print(
            f"SPAM {year} "
            f"{area_code}: "
            f"chunk {chunk_no} complete."
        )

    return qc

# SPAM 2020
# READ A / I / R INDEPENDENTLY TO NATIVE 5′ SPARSE COMPONENT FILES
def process_2020_component(
    area_code,
    technology,
    out_path,
):

    schema = (
        SOURCE_SCHEMA.loc[
            SOURCE_SCHEMA["year"].eq(2020)
            &
            SOURCE_SCHEMA["area_code"].eq(area_code)
            &
            SOURCE_SCHEMA["technology"].eq(technology)
        ]
        .iloc[0]
    )

    lookup = schema_crop_lookup(
        schema
    )

    expected = (
        SPAM_CROPS_BY_YEAR[
            2020
        ]
    )

    missing = (
        set(expected)
        -
        set(lookup)
    )

    if missing:

        raise ValueError(
            f"SPAM 2020 "
            f"{area_code} "
            f"T{technology}: "
            f"missing crops "
            f"{sorted(missing)}"
        )

    if out_path.exists():
        out_path.unlink()

    header_written = False

    qc = {
        "year": 2020,
        "area_code": area_code,
        "technology": technology,
        "rows_read": 0,
        "rows_written": 0,
        "missing_numeric_cells_to_zero": 0,
        "negative_cells_to_zero": 0,
    }

    for chunk_no, chunk in enumerate(
        make_reader(schema),
        start=1,
    ):

        keys = prepare_keys(
            chunk,
            schema,
        )

        qc[
            "rows_read"
        ] += len(chunk)

        for crop in expected:

            values, n_missing, n_negative = clean_numeric(
                chunk[
                    lookup[crop]
                ]
            )

            qc[
                "missing_numeric_cells_to_zero"
            ] += n_missing

            qc[
                "negative_cells_to_zero"
            ] += n_negative

            active = values.ne(0)

            if not active.any():
                continue

            mask = active.to_numpy(
                dtype=bool
            )

            part = (
                keys
                .iloc[mask]
                .copy()
            )

            part[
                "source_product_code"
            ] = crop

            part[
                "area_ha"
            ] = values.to_numpy()[mask]

            header_written = append_gzip_csv(
                part,
                out_path,
                header_written,
            )

            qc[
                "rows_written"
            ] += len(part)

        print(
            f"SPAM 2020 "
            f"{area_code} "
            f"T{technology}: "
            f"chunk {chunk_no} complete."
        )

    return qc

# SPAM 2020
# FINAL NATIVE 5′ KEY-BASED RECONCILIATION
def process_2020_area(
    area_code,
    out_path,
):

    component_paths = {
        "A":
            TEMP_DIR
            / f"_2020_{area_code}_TA_sparse.csv.gz",

        "I":
            TEMP_DIR
            / f"_2020_{area_code}_TI_sparse.csv.gz",

        "R":
            TEMP_DIR
            / f"_2020_{area_code}_TR_sparse.csv.gz",
    }

    component_qc = []

    # 1. BUILD OR REUSE COMPONENT CACHE
    for tech in [
        "A",
        "I",
        "R",
    ]:

        path = component_paths[tech]

        if (
            path.exists()
            and path.stat().st_size > 0
        ):

            print(
                f"SPAM 2020 {area_code} "
                f"T{tech}: existing sparse component REUSED."
            )

        else:

            q = process_2020_component(
                area_code=area_code,
                technology=tech,
                out_path=path,
            )

            component_qc.append(q)

    # 2. FULL UNION OF POSITIVE SOURCE KEYS
    reconciled_sql = f"""

    WITH

    stacked AS (

        SELECT
            grid_id_5min,
            grid_row_5min,
            grid_col_5min,
            longitude,
            latitude,
            country_iso3,
            year,
            source_product_code,
            'A' AS component,
            area_ha

        FROM read_csv_auto(
            '{component_paths["A"].as_posix()}',
            header=true
        )

        UNION ALL

        SELECT
            grid_id_5min,
            grid_row_5min,
            grid_col_5min,
            longitude,
            latitude,
            country_iso3,
            year,
            source_product_code,
            'I' AS component,
            area_ha

        FROM read_csv_auto(
            '{component_paths["I"].as_posix()}',
            header=true
        )

        UNION ALL

        SELECT
            grid_id_5min,
            grid_row_5min,
            grid_col_5min,
            longitude,
            latitude,
            country_iso3,
            year,
            source_product_code,
            'R' AS component,
            area_ha

        FROM read_csv_auto(
            '{component_paths["R"].as_posix()}',
            header=true
        )
    ),

    raw AS (

        SELECT

            grid_id_5min,

            MIN(grid_row_5min)
                AS grid_row_5min,

            MIN(grid_col_5min)
                AS grid_col_5min,

            MIN(longitude)
                AS longitude,

            MIN(latitude)
                AS latitude,

            country_iso3,
            year,
            source_product_code,

            SUM(
                CASE
                    WHEN component = 'A'
                    THEN area_ha
                    ELSE NULL
                END
            )
                AS total_raw_ha,

            SUM(
                CASE
                    WHEN component = 'I'
                    THEN area_ha
                    ELSE NULL
                END
            )
                AS irrigated_raw_ha,

            SUM(
                CASE
                    WHEN component = 'R'
                    THEN area_ha
                    ELSE NULL
                END
            )
                AS rainfed_raw_ha

        FROM stacked

        GROUP BY
            grid_id_5min,
            country_iso3,
            year,
            source_product_code
    ),

    resolved AS (

        SELECT

            *,

            -- Sparse component absence:
            -- no positive value exists in that component source.
            COALESCE(
                irrigated_raw_ha,
                0.0
            )
                AS irrigated_resolved_ha,

            COALESCE(
                rainfed_raw_ha,
                0.0
            )
                AS rainfed_resolved_ha

        FROM raw
    ),

    balanced AS (

        SELECT

            *,

            -- Working split total
            irrigated_resolved_ha
            +
            rainfed_resolved_ha
                AS total_computed_ha,

            -- Difference from published TA when TA exists
            CASE

                WHEN total_raw_ha IS NOT NULL

                    THEN
                        total_raw_ha
                        -
                        irrigated_resolved_ha
                        -
                        rainfed_resolved_ha

                ELSE NULL

            END
                AS balance_residual_ha

        FROM resolved
    ),

    classified AS (

        SELECT

            *,

            CASE

                -- ---------------------------------------------------------
                -- TA absent: reconstruct total from available split source.
                -- ---------------------------------------------------------

                WHEN total_raw_ha IS NULL

                 AND irrigated_raw_ha IS NULL

                 AND rainfed_raw_ha IS NOT NULL

                    THEN
                    'SOURCE_TOTAL_RECONSTRUCTED_FROM_RF_ONLY'

                WHEN total_raw_ha IS NULL

                 AND irrigated_raw_ha IS NOT NULL

                 AND rainfed_raw_ha IS NULL

                    THEN
                    'SOURCE_TOTAL_RECONSTRUCTED_FROM_IR_ONLY'

                WHEN total_raw_ha IS NULL

                 AND irrigated_raw_ha IS NOT NULL

                 AND rainfed_raw_ha IS NOT NULL

                    THEN
                    'SOURCE_TOTAL_RECONSTRUCTED_FROM_IR_PLUS_RF'

                -- ---------------------------------------------------------
                -- TA exists, both sparse split components absent.
                -- Diagnostic evidence: these are 0.1-ha source records.
                -- Do not arbitrarily assign them to irrigated or rainfed.
                -- ---------------------------------------------------------

                WHEN total_raw_ha IS NOT NULL

                 AND irrigated_raw_ha IS NULL

                 AND rainfed_raw_ha IS NULL

                 AND ABS(
                     balance_residual_ha
                 ) <= {SPAM2020_ROUNDING_ATOL_HA}

                    THEN
                    'SOURCE_COMPONENTS_SPARSE_ZERO_ROUNDING'

                -- ---------------------------------------------------------
                -- One split component absent and the TA difference is
                -- within the diagnosed ~0.1-ha source precision.
                -- ---------------------------------------------------------

                WHEN total_raw_ha IS NOT NULL

                 AND (
                        irrigated_raw_ha IS NULL
                        OR
                        rainfed_raw_ha IS NULL
                     )

                 AND ABS(
                     balance_residual_ha
                 ) <= {SPAM2020_ROUNDING_ATOL_HA}

                    THEN
                    'SOURCE_COMPONENTS_SPARSE_ZERO_ROUNDING'

                -- ---------------------------------------------------------
                -- All components present and essentially exact.
                -- ---------------------------------------------------------

                WHEN total_raw_ha IS NOT NULL

                 AND irrigated_raw_ha IS NOT NULL

                 AND rainfed_raw_ha IS NOT NULL

                 AND ABS(
                     balance_residual_ha
                 ) <= {AREA_ATOL_HA}

                    THEN
                    'SOURCE_COMPONENTS_COHERENT'

                -- ---------------------------------------------------------
                -- All components present and discrepancy is within
                -- diagnosed source reporting precision.
                -- ---------------------------------------------------------

                WHEN total_raw_ha IS NOT NULL

                 AND irrigated_raw_ha IS NOT NULL

                 AND rainfed_raw_ha IS NOT NULL

                 AND ABS(
                     balance_residual_ha
                 ) <= {SPAM2020_ROUNDING_ATOL_HA}

                    THEN
                    'SOURCE_COMPONENTS_COHERENT_ROUNDING'

                -- ---------------------------------------------------------
                -- Material source inconsistency.
                -- Retain but flag.
                -- ---------------------------------------------------------

                ELSE
                    'SOURCE_COMPONENT_REVIEW_REQUIRED'

            END
                AS source_component_status

        FROM balanced
    )

    SELECT

        *,

        source_component_status
            = 'SOURCE_COMPONENT_REVIEW_REQUIRED'
            AS manual_review_required,

        source_component_status IN (
            'SOURCE_COMPONENTS_COHERENT_ROUNDING',
            'SOURCE_COMPONENTS_SPARSE_ZERO_ROUNDING'
        )
            AS source_rounding_accepted,

        source_component_status IN (
            'SOURCE_TOTAL_RECONSTRUCTED_FROM_RF_ONLY',
            'SOURCE_TOTAL_RECONSTRUCTED_FROM_IR_ONLY',
            'SOURCE_TOTAL_RECONSTRUCTED_FROM_IR_PLUS_RF'
        )
            AS source_total_reconstructed

    FROM classified

    """

    # 3. STATUS QC
    status_qc = duckdb.sql(
        f"""

        WITH d AS (
            {reconciled_sql}
        )

        SELECT

            source_component_status,

            COUNT(*)
                AS rows,

            SUM(
                COALESCE(
                    total_raw_ha,
                    0.0
                )
            )
                AS source_TA_ha,

            SUM(
                irrigated_resolved_ha
            )
                AS irrigated_ha,

            SUM(
                rainfed_resolved_ha
            )
                AS rainfed_ha,

            SUM(
                total_computed_ha
            )
                AS computed_total_ha,

            SUM(
                COALESCE(
                    balance_residual_ha,
                    0.0
                )
            )
                AS signed_residual_ha,

            SUM(
                ABS(
                    COALESCE(
                        balance_residual_ha,
                        0.0
                    )
                )
            )
                AS absolute_residual_ha,

            MAX(
                ABS(
                    COALESCE(
                        balance_residual_ha,
                        0.0
                    )
                )
            )
                AS max_abs_residual_ha

        FROM d

        GROUP BY
            source_component_status

        ORDER BY
            source_component_status

        """
    ).df()

    status_qc.insert(
        0,
        "year",
        2020,
    )

    status_qc.insert(
        1,
        "area_code",
        area_code,
    )

    status_qc.to_csv(
        QC_DIR
        / f"01_SPAM2020_{area_code}_component_status_FINAL.csv",
        index=False,
    )

    print(
        f"\nSPAM 2020 {area_code} — FINAL component status:"
    )

    display(
        status_qc
    )

    # 4. MATERIAL REVIEW AUDIT
    # Review rows are retained, not silently changed.
    review = duckdb.sql(
        f"""

        WITH d AS (
            {reconciled_sql}
        )

        SELECT

            grid_id_5min,
            grid_row_5min,
            grid_col_5min,
            longitude,
            latitude,
            country_iso3,
            year,
            source_product_code,

            total_raw_ha,
            irrigated_raw_ha,
            rainfed_raw_ha,

            irrigated_resolved_ha,
            rainfed_resolved_ha,
            total_computed_ha,

            balance_residual_ha,

            source_component_status,
            manual_review_required

        FROM d

        WHERE
            manual_review_required

        ORDER BY
            ABS(
                balance_residual_ha
            ) DESC,
            country_iso3,
            source_product_code,
            grid_id_5min

        """
    ).df()

    review_path = (
        QC_DIR
        / f"01_SPAM2020_{area_code}_SOURCE_COMPONENT_REVIEW_REQUIRED_FINAL.csv"
    )

    review.to_csv(
        review_path,
        index=False,
    )

    if review.empty:

        print(
            f"SPAM 2020 {area_code}: "
            "material source-component review rows = 0."
        )

    else:

        print(
            f"SPAM 2020 {area_code}: "
            f"{len(review):,} material source-component "
            "review rows retained and flagged."
        )

        print(
            "Review audit:",
            review_path,
        )

    # 5. ROUNDING / TOTAL-RECONSTRUCTION AUDIT
    audit = duckdb.sql(
        f"""

        WITH d AS (
            {reconciled_sql}
        )

        SELECT

            grid_id_5min,
            grid_row_5min,
            grid_col_5min,
            longitude,
            latitude,
            country_iso3,
            year,
            source_product_code,

            total_raw_ha,
            irrigated_raw_ha,
            rainfed_raw_ha,

            irrigated_resolved_ha,
            rainfed_resolved_ha,
            total_computed_ha,

            balance_residual_ha,

            source_component_status,
            source_rounding_accepted,
            source_total_reconstructed

        FROM d

        WHERE
            source_rounding_accepted
            OR
            source_total_reconstructed

        ORDER BY
            source_component_status,
            country_iso3,
            source_product_code,
            grid_id_5min

        """
    ).df()

    audit.to_csv(
        QC_DIR
        / f"01_SPAM2020_{area_code}_reconciliation_audit_FINAL.csv",
        index=False,
    )

    # 6. WRITE FINAL 2020 SPARSE FILE
    if out_path.exists():
        out_path.unlink()

    duckdb.sql(
        f"""

        COPY (

            WITH d AS (
                {reconciled_sql}
            )

            SELECT

                grid_id_5min,
                grid_row_5min,
                grid_col_5min,
                longitude,
                latitude,
                country_iso3,
                year,
                source_product_code,

                -- Original SPAM total.
                -- NULL when TA itself was absent.
                total_raw_ha
                    AS source_total_source_ha,

                -- Published sparse split values.
                irrigated_resolved_ha
                    AS source_irrigated_ha,

                rainfed_resolved_ha
                    AS source_rainfed_ha,

                -- Working split total.
                -- Exactly IR + RF.
                total_computed_ha
                    AS source_total_ha,

                -- Published TA minus working split total.
                balance_residual_ha
                    AS source_balance_residual_ha,

                source_component_status,
                manual_review_required,
                source_rounding_accepted,
                source_total_reconstructed

            FROM d

        )

        TO '{out_path.as_posix()}'

        (
            FORMAT CSV,
            HEADER TRUE,
            COMPRESSION GZIP
        );

        """
    )

    # 7. FINAL OUTPUT SUMMARY
    final_summary = duckdb.sql(
        f"""

        SELECT

            COUNT(*)
                AS rows,

            SUM(source_irrigated_ha)
                AS irrigated_ha,

            SUM(source_rainfed_ha)
                AS rainfed_ha,

            SUM(source_total_ha)
                AS computed_total_ha,

            SUM(
                COALESCE(
                    source_total_source_ha,
                    0.0
                )
            )
                AS original_TA_ha,

            SUM(
                CASE
                    WHEN manual_review_required
                    THEN 1
                    ELSE 0
                END
            )
                AS review_rows,

            SUM(
                CASE
                    WHEN source_rounding_accepted
                    THEN 1
                    ELSE 0
                END
            )
                AS rounding_rows,

            SUM(
                CASE
                    WHEN source_total_reconstructed
                    THEN 1
                    ELSE 0
                END
            )
                AS reconstructed_total_rows

        FROM read_csv_auto(
            '{out_path.as_posix()}',
            header=true
        )

        """
    ).df()

    print(
        f"\nSPAM 2020 {area_code} — FINAL sparse output:"
    )

    display(
        final_summary
    )

    final_rows = int(
        final_summary.loc[
            0,
            "rows",
        ]
    )

    review_rows = int(
        final_summary.loc[
            0,
            "review_rows",
        ]
    )

    rounding_rows = int(
        final_summary.loc[
            0,
            "rounding_rows",
        ]
    )

    reconstructed_rows = int(
        final_summary.loc[
            0,
            "reconstructed_total_rows",
        ]
    )

    # 8. RETURN PROCESSING QC
    if component_qc:

        rows_read = sum(
            x["rows_read"]
            for x in component_qc
        )

        missing_count = sum(
            x[
                "missing_numeric_cells_to_zero"
            ]
            for x in component_qc
        )

        negative_count = sum(
            x[
                "negative_cells_to_zero"
            ]
            for x in component_qc
        )

    else:

        # Sparse components were reused.
        rows_read = np.nan
        missing_count = np.nan
        negative_count = np.nan

    qc = {
        "year": 2020,
        "area_code": area_code,
        "reader_mode":
            "2020_KEY_BASED_NATIVE_5MIN_FINAL",
        "rows_read":
            rows_read,
        "rows_written":
            final_rows,
        "missing_numeric_cells_to_zero":
            missing_count,
        "negative_cells_to_zero":
            negative_count,
        "source_rounding_tolerance_ha":
            SPAM2020_ROUNDING_ATOL_HA,
        "source_rounding_accepted_rows":
            rounding_rows,
        "source_total_reconstructed_rows":
            reconstructed_rows,
        "source_component_review_rows":
            review_rows,
    }

    print(
        f"\nSPAM 2020 {area_code}: "
        f"FINAL reconciliation complete — "
        f"{final_rows:,} rows; "
        f"{rounding_rows:,} rounding-supported rows; "
        f"{reconstructed_rows:,} reconstructed-total rows; "
        f"{review_rows:,} source review rows retained."
    )

    return qc

# MASTER SPLIT-AREA DISPATCHER USED BY CELL 6
def process_split_area(
    year,
    area_code,
    out_path,
):

    if year in {
        2005,
        2010,
    }:

        return process_aligned_split_area(
            year,
            area_code,
            out_path,
        )

    if year == 2020:

        return process_2020_area(
            area_code,
            out_path,
        )

    raise ValueError(
        f"process_split_area does not support year={year}"
    )

print(
    "CELL 5 STATUS: READY — "
    "2000 embedded reader + "
    "2005/2010 aligned A/I/R reader + "
    "2020 native-5′ key-based sparse reconciliation."
)

## 06 | SPAM2000 OTHE physical-area rule

The later corrected source reader retains available OTHE total physical area even when the IR/RF split is missing. This definition supersedes the original reader for SPAM2000 only.

In [ ]:
# PURPOSE
# -------
# Override ONLY process_2000_area().
# SPAM2000 OTHE special source structure:
#   Harvested area:
#       no usable OTHE harvested magnitude from this source structure
#   Physical area:
#       source TOTAL may be positive
#       I / H / L / S split may all be zero / unavailable
# Previous Notebook01 logic computed:
#       source_total_ha = I + H + L + S
# which reduced those OTHE physical-total-only rows to zero downstream.
# CORRECTION
# Preserve:
#       source_total_source_ha = published SPAM total
# For the specific OTHE physical-total-only case:
#       source_irrigated_ha = NaN
#       source_rainfed_ha   = NaN
#       source_total_ha     = NaN
# because the IR/RF split is not actually available.
# A dedicated status flag records that the published total exists.
# IMPORTANT:
# Cell 6 must be corrected next so physical_total_ha uses the published
# source total for these flagged rows without inventing an IR/RF split.
# No other SPAM2000 crop is changed.
# No 2005/2010/2020 logic is changed.
def process_2000_area(
    area_code,
    out_path,
):

    area_code = str(
        area_code
    ).strip().upper()

    if area_code not in {
        "H",
        "A",
    }:

        raise ValueError(
            f"SPAM2000 unsupported area_code: {area_code}"
        )

    # 1. RESOLVE SOURCE SCHEMA
    schema = (
        SOURCE_SCHEMA.loc[
            SOURCE_SCHEMA[
                "year"
            ].eq(
                2000
            )
            &
            SOURCE_SCHEMA[
                "area_code"
            ].eq(
                area_code
            )
        ]
        .iloc[0]
    )

    lookup = schema_embedded_lookup(
        schema
    )

    required = {
        (
            crop,
            suffix,
        )

        for crop in SPAM_CROPS_2000

        for suffix in [
            "total",
            "h",
            "l",
            "i",
            "s",
        ]
    }

    if not required.issubset(
        set(
            lookup
        )
    ):

        raise ValueError(
            f"SPAM 2000 {area_code}: "
            "missing embedded crop columns."
        )

    # 2. CLEAN OLD SPARSE OUTPUT
    if out_path.exists():

        out_path.unlink()

    header_written = False

    # 3. QC
    qc = {

        "year":
            2000,

        "area_code":
            area_code,

        "reader_mode":
            "2000_EMBEDDED_HLIS_OTHE_TOTAL_ONLY_CORRECTED",

        "rows_read":
            0,

        "missing_numeric_cells_to_zero":
            0,

        "negative_cells_to_zero":
            0,

        "rows_written":
            0,

        "othe_total_only_rows":
            0,

        "othe_total_only_source_area_ha":
            0.0,

        "othe_split_nonzero_rows":
            0,

    }

    # 4. PROCESS SOURCE CHUNKS
    for chunk_no, chunk in enumerate(
        make_reader(
            schema
        ),
        start=1,
    ):

        keys = prepare_keys(
            chunk,
            schema,
        )

        qc[
            "rows_read"
        ] += len(
            chunk
        )

        # EACH NATIVE SPAM2000 CROP
        for crop in SPAM_CROPS_2000:

            # Published total
            total, m0, n0 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "total",
                        )
                    ]
                ]
            )

            # Irrigated component
            irrigated, m1, n1 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "i",
                        )
                    ]
                ]
            )

            # Rainfed production systems
            h, m2, n2 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "h",
                        )
                    ]
                ]
            )

            l, m3, n3 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "l",
                        )
                    ]
                ]
            )

            s, m4, n4 = clean_numeric(
                chunk[
                    lookup[
                        (
                            crop,
                            "s",
                        )
                    ]
                ]
            )

            rainfed = (
                h
                +
                l
                +
                s
            )

            computed = (
                irrigated
                +
                rainfed
            )

            # Parsing QC
            qc[
                "missing_numeric_cells_to_zero"
            ] += (
                m0
                +
                m1
                +
                m2
                +
                m3
                +
                m4
            )

            qc[
                "negative_cells_to_zero"
            ] += (
                n0
                +
                n1
                +
                n2
                +
                n3
                +
                n4
            )

            # 5. IDENTIFY SPAM2000 OTHE PHYSICAL-TOTAL-ONLY CASE
            # Only physical-area OTHE rows are eligible for this exception.
            # We require:
            #   published total > 0
            #   AND
            #   I + H + L + S == 0
            # If SPAM unexpectedly contains a non-zero split for an OTHE row,
            # that row is NOT treated as total-only.
            othe_total_only = (
                area_code
                ==
                "A"
            ) & (
                crop
                ==
                "othe"
            )

            if othe_total_only:

                total_only_mask = (
                    total.gt(
                        0
                    )
                    &
                    np.isclose(
                        computed.to_numpy(
                            dtype=float
                        ),
                        0.0,
                        rtol=0.0,
                        atol=AREA_ATOL_HA,
                    )
                )

                split_nonzero_mask = (
                    total.gt(
                        0
                    )
                    &
                    ~total_only_mask
                )

                qc[
                    "othe_total_only_rows"
                ] += int(
                    total_only_mask.sum()
                )

                qc[
                    "othe_total_only_source_area_ha"
                ] += float(
                    total.loc[
                        total_only_mask
                    ]
                    .sum()
                )

                qc[
                    "othe_split_nonzero_rows"
                ] += int(
                    split_nonzero_mask.sum()
                )

            else:

                total_only_mask = pd.Series(
                    False,
                    index=total.index,
                    dtype=bool,
                )

            # 6. ACTIVE ROWS
            active = (
                total.ne(
                    0
                )
                |
                irrigated.ne(
                    0
                )
                |
                rainfed.ne(
                    0
                )
            )

            if not active.any():

                continue

            # IMPORTANT:
            # positional mask avoids chunk-index alignment problems.

            mask = active.to_numpy(
                dtype=bool
            )

            part = (
                keys
                .iloc[
                    mask
                ]
                .copy()
            )

            part[
                "source_product_code"
            ] = crop

            total_active = (
                total.to_numpy(
                    dtype=float
                )[
                    mask
                ]
            )

            irrigated_active = (
                irrigated.to_numpy(
                    dtype=float
                )[
                    mask
                ]
            )

            rainfed_active = (
                rainfed.to_numpy(
                    dtype=float
                )[
                    mask
                ]
            )

            computed_active = (
                computed.to_numpy(
                    dtype=float
                )[
                    mask
                ]
            )

            total_only_active = (
                total_only_mask.to_numpy(
                    dtype=bool
                )[
                    mask
                ]
            )

            # 7. STANDARD SOURCE VALUES
            part[
                "source_total_source_ha"
            ] = total_active

            part[
                "source_irrigated_ha"
            ] = irrigated_active

            part[
                "source_rainfed_ha"
            ] = rainfed_active

            part[
                "source_total_ha"
            ] = computed_active

            part[
                "source_balance_residual_ha"
            ] = (
                total_active
                -
                computed_active
            )

            part[
                "source_total_only_available"
            ] = False

            part[
                "source_split_available"
            ] = True

            part[
                "source_component_status"
            ] = (
                "SOURCE_COMPONENTS_AVAILABLE"
            )

            # 8. OTHE PHYSICAL-TOTAL-ONLY CORRECTION
            # DO NOT set IR or RF to zero.
            # Those components are unavailable, not genuine zero estimates.
            # DO NOT manufacture a split.
            # Keep the published source total separately and mark the split
            # explicitly unavailable.
            if total_only_active.any():

                idx = part.index[
                    total_only_active
                ]

                part.loc[
                    idx,
                    "source_irrigated_ha",
                ] = np.nan

                part.loc[
                    idx,
                    "source_rainfed_ha",
                ] = np.nan

                part.loc[
                    idx,
                    "source_total_ha",
                ] = np.nan

                part.loc[
                    idx,
                    "source_balance_residual_ha",
                ] = np.nan

                part.loc[
                    idx,
                    "source_total_only_available",
                ] = True

                part.loc[
                    idx,
                    "source_split_available",
                ] = False

                part.loc[
                    idx,
                    "source_component_status",
                ] = (
                    "SOURCE_TOTAL_ONLY_IR_RF_SPLIT_UNAVAILABLE"
                )

            # 9. WRITE SPARSE ROWS
            header_written = append_gzip_csv(
                part,
                out_path,
                header_written,
            )

            qc[
                "rows_written"
            ] += len(
                part
            )

        print(
            f"SPAM 2000 {area_code}: "
            f"chunk {chunk_no} complete."
        )

    # 10. FINAL OTHE QC
    if area_code == "A":

        print(
            "\n=== SPAM2000 OTHE PHYSICAL SOURCE-TOTAL QC ==="
        )

        print(
            "OTHE total-only rows:",
            f"{qc['othe_total_only_rows']:,}",
        )

        print(
            "OTHE published physical total preserved (ha):",
            f"{qc['othe_total_only_source_area_ha']:,.3f}",
        )

        print(
            "OTHE rows with unexpected non-zero split:",
            f"{qc['othe_split_nonzero_rows']:,}",
        )

        if (
            qc[
                "othe_total_only_rows"
            ]
            ==
            0
        ):

            raise RuntimeError(
                "No positive SPAM2000 OTHE physical-total-only rows "
                "were detected. Stop and inspect the raw source before "
                "continuing."
            )

        if (
            qc[
                "othe_total_only_source_area_ha"
            ]
            <=
            0
        ):

            raise RuntimeError(
                "SPAM2000 OTHE source physical total is not positive."
            )

    # 11. RETURN
    return qc

print(
    "CELL 5A STATUS: READY — "
    "SPAM2000 OTHE published physical total will be preserved "
    "with IR/RF split explicitly unavailable."
)

## 07 | Historical SPAM2000 OTHE repair

**Existing-output repair, not a clean-start production stage.** This code makes a backup and updates an existing 2000 Parquet file after rebuilding the 2000 sparse source. It is preserved for traceability until integrated into a unified initial production runner.

In [ ]:
# OTHE PHYSICAL TOTAL-ONLY SOURCE PRESERVATION
# PURPOSE
# -------
# Rebuild ONLY the 2000 native SPAM asset after Cell 5A.
# Special SPAM2000 OTHE rule:
#   published physical total exists
#   but IR / RF physical split is unavailable
# Therefore:
#   physical_total_ha      = published SPAM source total
#   physical_irrigated_ha  = NULL
#   physical_rainfed_ha    = NULL
# No split is invented.
# 2005 / 2010 / 2020 are NOT rebuilt.
# Main manifest and country QC are NOT updated yet.
from datetime import datetime
import shutil

# 1. SETTINGS
REBUILD_YEAR = 2000

if "AREA_ATOL_HA" not in globals():
    AREA_ATOL_HA = 1e-4

# 2. PATHS
sparse_h = (
    TEMP_DIR
    / "SPAM_2000_harvested_sparse_5arcmin.csv.gz"
)

sparse_p = (
    TEMP_DIR
    / "SPAM_2000_physical_sparse_5arcmin.csv.gz"
)

final_parquet = (
    OUTPUT_DIR
    / "01_SPAM_native_5arcmin_2000.parquet"
)

# 3. BACKUP CURRENT 2000 NATIVE ASSET
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

backup_dir = (
    OUTPUT_DIR
    / "_repair_backup"
    / f"pre_OTHE_physical_fix_{timestamp}"
)

backup_dir.mkdir(
    parents=True,
    exist_ok=False,
)

backup_parquet = (
    backup_dir
    / final_parquet.name
)

if final_parquet.exists():

    shutil.copy2(
        final_parquet,
        backup_parquet,
    )

    print(
        "Existing 2000 native asset backed up to:"
    )

    print(
        backup_parquet
    )

else:

    raise FileNotFoundError(
        final_parquet
    )

# 4. REBUILD 2000 HARVESTED + PHYSICAL SPARSE SOURCE TABLES
print(
    "\n" + "=" * 80
)

print(
    "PROCESSING SPAM 2000 — HARVESTED AREA"
)

print(
    "=" * 80
)

qc_h_2000 = process_2000_area(
    "H",
    sparse_h,
)

print(
    "\n" + "=" * 80
)

print(
    "PROCESSING SPAM 2000 — PHYSICAL AREA"
)

print(
    "=" * 80
)

qc_p_2000 = process_2000_area(
    "A",
    sparse_p,
)

PROCESSING_QC_2000_REPAIR = pd.DataFrame(
    [
        qc_h_2000,
        qc_p_2000,
    ]
)

print(
    "\n=== CELL 6 RAW PROCESSING QC ==="
)

display(
    PROCESSING_QC_2000_REPAIR
)

# 5. PRE-PARQUET OTHE SOURCE CHECK
con = duckdb.connect()

try:

    OTHE_SPARSE_SOURCE_QC = con.execute(
        f"""
        SELECT

            COUNT(*)
                AS source_rows,

            COUNT(
                DISTINCT country_iso3
            )
                AS countries,

            SUM(
                source_total_source_ha
            )
                AS published_physical_total_ha,

            SUM(
                source_irrigated_ha
            )
                AS available_physical_irrigated_ha,

            SUM(
                source_rainfed_ha
            )
                AS available_physical_rainfed_ha,

            SUM(
                CASE
                    WHEN TRY_CAST(
                        source_total_only_available
                        AS BOOLEAN
                    )
                    THEN 1
                    ELSE 0
                END
            )
                AS total_only_rows,

            SUM(
                CASE
                    WHEN TRY_CAST(
                        source_split_available
                        AS BOOLEAN
                    )
                    THEN 1
                    ELSE 0
                END
            )
                AS split_available_rows

        FROM read_csv_auto(
            '{sparse_p.as_posix()}',
            header = true
        )

        WHERE lower(
            trim(
                source_product_code
            )
        ) = 'othe'
        """
    ).df()

finally:

    con.close()

print(
    "\n=== SPAM2000 OTHE SOURCE PHYSICAL-AREA CHECK ==="
)

display(
    OTHE_SPARSE_SOURCE_QC
)

if (
    OTHE_SPARSE_SOURCE_QC[
        "published_physical_total_ha"
    ]
    .iloc[0]
    <=
    0
):

    raise RuntimeError(
        "OTHE published physical total is still zero. "
        "Do not continue."
    )

# 6. BUILD CORRECTED 2000 NATIVE PARQUET
con = duckdb.connect()

try:

    con.execute(
        f"""
        COPY
        (
            WITH

            h AS
            (
                SELECT

                    grid_id_5min,
                    grid_row_5min,
                    grid_col_5min,
                    longitude,
                    latitude,

                    country_iso3,
                    year,
                    source_product_code,

                    SUM(
                        source_total_source_ha
                    )
                        AS h_total_source,

                    SUM(
                        source_irrigated_ha
                    )
                        AS h_ir,

                    SUM(
                        source_rainfed_ha
                    )
                        AS h_rf,

                    SUM(
                        source_total_ha
                    )
                        AS h_total,

                    SUM(
                        source_balance_residual_ha
                    )
                        AS h_residual

                FROM read_csv_auto(
                    '{sparse_h.as_posix()}',
                    header = true
                )

                GROUP BY ALL
            ),

            p AS
            (
                SELECT

                    grid_id_5min,
                    grid_row_5min,
                    grid_col_5min,
                    longitude,
                    latitude,

                    country_iso3,
                    year,
                    source_product_code,

                    SUM(
                        source_total_source_ha
                    )
                        AS p_total_source,

                    SUM(
                        source_irrigated_ha
                    )
                        AS p_ir,

                    SUM(
                        source_rainfed_ha
                    )
                        AS p_rf,

                    SUM(
                        source_total_ha
                    )
                        AS p_total,

                    SUM(
                        source_balance_residual_ha
                    )
                        AS p_residual,

                    MAX(
                        CASE
                            WHEN TRY_CAST(
                                source_total_only_available
                                AS BOOLEAN
                            )
                            THEN 1
                            ELSE 0
                        END
                    )
                        AS p_total_only_available

                FROM read_csv_auto(
                    '{sparse_p.as_posix()}',
                    header = true
                )

                GROUP BY ALL
            ),

            joined AS
            (
                SELECT

                    COALESCE(
                        h.grid_id_5min,
                        p.grid_id_5min
                    )
                        AS grid_id_5min,

                    COALESCE(
                        h.grid_row_5min,
                        p.grid_row_5min
                    )
                        AS grid_row_5min,

                    COALESCE(
                        h.grid_col_5min,
                        p.grid_col_5min
                    )
                        AS grid_col_5min,

                    COALESCE(
                        h.longitude,
                        p.longitude
                    )
                        AS longitude,

                    COALESCE(
                        h.latitude,
                        p.latitude
                    )
                        AS latitude,

                    COALESCE(
                        h.country_iso3,
                        p.country_iso3
                    )
                        AS country_iso3,

                    COALESCE(
                        h.year,
                        p.year
                    )
                        AS year,

                    COALESCE(
                        h.source_product_code,
                        p.source_product_code
                    )
                        AS source_product_code,

                    -- =========================================================
                    -- HARVESTED AREA
                    -- =========================================================

                    COALESCE(
                        h.h_ir,
                        0.0
                    )
                        AS harvested_irrigated_ha,

                    COALESCE(
                        h.h_rf,
                        0.0
                    )
                        AS harvested_rainfed_ha,

                    COALESCE(
                        h.h_total,
                        0.0
                    )
                        AS harvested_total_ha,

                    COALESCE(
                        h.h_total_source,
                        0.0
                    )
                        AS harvested_total_source_ha,

                    COALESCE(
                        h.h_residual,
                        0.0
                    )
                        AS harvested_balance_residual_ha,

                    -- =========================================================
                    -- PHYSICAL AREA
                    -- =========================================================
                    --
                    -- Standard crops:
                    --     preserve old behaviour.
                    --
                    -- OTHE total-only rows:
                    --     IR/RF = NULL
                    --     total = published source total.
                    -- =========================================================

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN CAST(
                            NULL AS DOUBLE
                        )

                        ELSE COALESCE(
                            p.p_ir,
                            0.0
                        )

                    END
                        AS physical_irrigated_ha,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN CAST(
                            NULL AS DOUBLE
                        )

                        ELSE COALESCE(
                            p.p_rf,
                            0.0
                        )

                    END
                        AS physical_rainfed_ha,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN p.p_total_source

                        ELSE COALESCE(
                            p.p_total,
                            0.0
                        )

                    END
                        AS physical_total_ha,

                    COALESCE(
                        p.p_total_source,
                        0.0
                    )
                        AS physical_total_source_ha,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN CAST(
                            NULL AS DOUBLE
                        )

                        ELSE COALESCE(
                            p.p_residual,
                            0.0
                        )

                    END
                        AS physical_balance_residual_ha,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN TRUE

                        ELSE FALSE

                    END
                        AS physical_total_only_available,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN FALSE

                        ELSE TRUE

                    END
                        AS physical_split_available,

                    CASE

                        WHEN COALESCE(
                            p.p_total_only_available,
                            0
                        ) = 1

                        THEN
                            'SOURCE_TOTAL_ONLY_IR_RF_SPLIT_UNAVAILABLE'

                        ELSE
                            'SOURCE_COMPONENTS_AVAILABLE'

                    END
                        AS physical_component_status

                FROM h

                FULL OUTER JOIN p

                USING
                (
                    grid_id_5min,
                    grid_row_5min,
                    grid_col_5min,
                    longitude,
                    latitude,
                    country_iso3,
                    year,
                    source_product_code
                )
            )

            SELECT

                *,

                CASE

                    WHEN physical_irrigated_ha > 0

                    THEN
                        harvested_irrigated_ha
                        /
                        physical_irrigated_ha

                    ELSE NULL

                END
                    AS harvesting_intensity_irrigated,

                CASE

                    WHEN physical_rainfed_ha > 0

                    THEN
                        harvested_rainfed_ha
                        /
                        physical_rainfed_ha

                    ELSE NULL

                END
                    AS harvesting_intensity_rainfed,

                CASE

                    WHEN physical_total_only_available

                    THEN NULL

                    WHEN physical_total_ha > 0

                    THEN
                        harvested_total_ha
                        /
                        physical_total_ha

                    ELSE NULL

                END
                    AS harvesting_intensity_total,

                'SPAM'
                    AS source_dataset,

                'observed_SPAM'
                    AS temporal_status

            FROM joined
        )

        TO '{final_parquet.as_posix()}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

finally:

    con.close()

# 7. VERIFY FINAL OTHE VALUES
con = duckdb.connect()

try:

    OTHE_FINAL_QC = con.execute(
        f"""
        SELECT

            COUNT(*)
                AS final_rows,

            COUNT(
                DISTINCT country_iso3
            )
                AS countries,

            SUM(
                physical_total_source_ha
            )
                AS physical_total_source_ha,

            SUM(
                physical_total_ha
            )
                AS physical_total_ha,

            SUM(
                CASE
                    WHEN physical_total_only_available
                    THEN 1
                    ELSE 0
                END
            )
                AS total_only_rows,

            SUM(
                CASE
                    WHEN
                        physical_total_only_available
                        AND
                        physical_irrigated_ha IS NULL
                        AND
                        physical_rainfed_ha IS NULL

                    THEN 1
                    ELSE 0
                END
            )
                AS correctly_unresolved_split_rows,

            SUM(
                CASE
                    WHEN
                        physical_total_only_available
                        AND
                        (
                            physical_irrigated_ha IS NOT NULL
                            OR
                            physical_rainfed_ha IS NOT NULL
                        )

                    THEN 1
                    ELSE 0
                END
            )
                AS incorrectly_assigned_split_rows

        FROM read_parquet(
            '{final_parquet.as_posix()}'
        )

        WHERE lower(
            trim(
                source_product_code
            )
        ) = 'othe'
        """
    ).df()

finally:

    con.close()

print(
    "\n=== CORRECTED FINAL SPAM2000 OTHE QC ==="
)

display(
    OTHE_FINAL_QC
)

# 8. SOURCE → FINAL OTHE CONSERVATION
source_othe_total = float(
    OTHE_SPARSE_SOURCE_QC[
        "published_physical_total_ha"
    ]
    .iloc[0]
)

final_othe_total = float(
    OTHE_FINAL_QC[
        "physical_total_ha"
    ]
    .iloc[0]
)

OTHE_TOTAL_CONSERVATION_PASS = bool(
    np.isclose(
        source_othe_total,
        final_othe_total,
        rtol=1e-10,
        atol=1e-4,
    )
)

OTHE_SPLIT_STATUS_PASS = bool(
    int(
        OTHE_FINAL_QC[
            "incorrectly_assigned_split_rows"
        ]
        .iloc[0]
    )
    ==
    0
)

OTHE_TOTAL_PRESENT_PASS = bool(
    final_othe_total
    >
    0
)

# 9. VERIFY ALL NON-OTHE CROPS REMAIN NUMERICALLY UNCHANGED
con = duckdb.connect()

try:

    NON_OTHE_OLD = con.execute(
        f"""
        SELECT

            source_product_code,

            COUNT(*)
                AS rows,

            SUM(
                harvested_irrigated_ha
            )
                AS harvested_irrigated_ha,

            SUM(
                harvested_rainfed_ha
            )
                AS harvested_rainfed_ha,

            SUM(
                harvested_total_ha
            )
                AS harvested_total_ha,

            SUM(
                physical_irrigated_ha
            )
                AS physical_irrigated_ha,

            SUM(
                physical_rainfed_ha
            )
                AS physical_rainfed_ha,

            SUM(
                physical_total_ha
            )
                AS physical_total_ha

        FROM read_parquet(
            '{backup_parquet.as_posix()}'
        )

        WHERE lower(
            trim(
                source_product_code
            )
        ) <> 'othe'

        GROUP BY source_product_code
        """
    ).df()

    NON_OTHE_NEW = con.execute(
        f"""
        SELECT

            source_product_code,

            COUNT(*)
                AS rows,

            SUM(
                harvested_irrigated_ha
            )
                AS harvested_irrigated_ha,

            SUM(
                harvested_rainfed_ha
            )
                AS harvested_rainfed_ha,

            SUM(
                harvested_total_ha
            )
                AS harvested_total_ha,

            SUM(
                physical_irrigated_ha
            )
                AS physical_irrigated_ha,

            SUM(
                physical_rainfed_ha
            )
                AS physical_rainfed_ha,

            SUM(
                physical_total_ha
            )
                AS physical_total_ha

        FROM read_parquet(
            '{final_parquet.as_posix()}'
        )

        WHERE lower(
            trim(
                source_product_code
            )
        ) <> 'othe'

        GROUP BY source_product_code
        """
    ).df()

finally:

    con.close()

NON_OTHE_COMPARE = (
    NON_OTHE_OLD
    .merge(
        NON_OTHE_NEW,

        on="source_product_code",

        how="outer",

        suffixes=(
            "_old",
            "_new",
        ),

        indicator=True,
    )
)

compare_area_cols = [
    "harvested_irrigated_ha",
    "harvested_rainfed_ha",
    "harvested_total_ha",
    "physical_irrigated_ha",
    "physical_rainfed_ha",
    "physical_total_ha",
]

for col in compare_area_cols:

    NON_OTHE_COMPARE[
        f"diff_{col}"
    ] = (
        NON_OTHE_COMPARE[
            f"{col}_new"
        ]
        -
        NON_OTHE_COMPARE[
            f"{col}_old"
        ]
    )

NON_OTHE_COMPARE[
    "max_abs_area_difference_ha"
] = (
    NON_OTHE_COMPARE[
        [
            f"diff_{col}"
            for col in compare_area_cols
        ]
    ]
    .abs()
    .max(
        axis=1
    )
)

NON_OTHE_COMPARE[
    "row_count_equal"
] = (
    NON_OTHE_COMPARE[
        "rows_old"
    ]
    ==
    NON_OTHE_COMPARE[
        "rows_new"
    ]
)

NON_OTHE_COMPARE[
    "values_equal"
] = (
    NON_OTHE_COMPARE[
        "max_abs_area_difference_ha"
    ]
    .le(
        1e-4
    )
)

NON_OTHE_UNCHANGED_PASS = bool(
    NON_OTHE_COMPARE[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()

    and

    NON_OTHE_COMPARE[
        "row_count_equal"
    ]
    .all()

    and

    NON_OTHE_COMPARE[
        "values_equal"
    ]
    .all()
)

print(
    "\n=== NON-OTHE 2000 CROPS — OLD vs NEW ==="
)

display(
    NON_OTHE_COMPARE[
        [
            "source_product_code",
            "rows_old",
            "rows_new",
            "row_count_equal",
            "max_abs_area_difference_ha",
            "values_equal",
        ]
    ]
    .sort_values(
        "source_product_code"
    )
)

# 10. FINAL REPAIR STATUS
CELL6_PASS = bool(
    OTHE_TOTAL_CONSERVATION_PASS
    and
    OTHE_SPLIT_STATUS_PASS
    and
    OTHE_TOTAL_PRESENT_PASS
    and
    NON_OTHE_UNCHANGED_PASS
)

CELL6_STATUS = pd.DataFrame(
    [
        {
            "year":
                2000,

            "OTHE_source_physical_total_ha":
                source_othe_total,

            "OTHE_final_physical_total_ha":
                final_othe_total,

            "OTHE_total_conservation_pass":
                OTHE_TOTAL_CONSERVATION_PASS,

            "OTHE_IR_RF_split_left_unresolved":
                OTHE_SPLIT_STATUS_PASS,

            "OTHE_physical_total_present":
                OTHE_TOTAL_PRESENT_PASS,

            "all_non_OTHE_crops_unchanged":
                NON_OTHE_UNCHANGED_PASS,

            "other_years_rebuilt":
                False,

            "status":
                (
                    "PASS_SPAM2000_OTHE_PHYSICAL_TOTAL_RESTORED"
                    if CELL6_PASS
                    else
                    "REVIEW_REQUIRED"
                ),
        }
    ]
)

print(
    "\n=== CELL 6 FINAL STATUS ==="
)

display(
    CELL6_STATUS
)

# 11. SAVE REPAIR QC ONLY
PROCESSING_QC_2000_REPAIR.to_csv(
    QC_DIR
    / "01_REPAIR_2000_OTHE_raw_processing_QC.csv",

    index=False,
)

OTHE_SPARSE_SOURCE_QC.to_csv(
    QC_DIR
    / "01_REPAIR_2000_OTHE_source_physical_QC.csv",

    index=False,
)

OTHE_FINAL_QC.to_csv(
    QC_DIR
    / "01_REPAIR_2000_OTHE_final_physical_QC.csv",

    index=False,
)

NON_OTHE_COMPARE.to_csv(
    QC_DIR
    / "01_REPAIR_2000_non_OTHE_unchanged_QC.csv",

    index=False,
)

CELL6_STATUS.to_csv(
    QC_DIR
    / "01_REPAIR_2000_OTHE_status.csv",

    index=False,
)

# 12. STOP IF NOT VERIFIED
if not CELL6_PASS:

    raise RuntimeError(
        "SPAM2000 OTHE repair did not pass. "
        "Do not continue to Cell 7."
    )

print(
    "\nCELL 6 STATUS: PASS_SPAM2000_OTHE_PHYSICAL_TOTAL_RESTORED"
)

print(
    "\nIMPORTANT:"
)

print(
    "Do NOT run the old Cell 7 yet."
)

print(
    "Next step is a corrected 2000-aware country QC / source-total verification."
)

## 08 | Country-level audit and downstream handoff

Country-level checks reconcile the native Parquet outputs and update the country totals used by Notebook 02. A full rerun requires all four initial Parquet outputs to exist.

In [ ]:
# PURPOSE
# -------
# 1. Rebuild country × native-SPAM-crop QC for ALL observed years
#    from the current native 5′ parquet files.
# 2. Preserve the special SPAM2000 OTHE structure:
#       physical_total_ha      = available source total
#       physical_irrigated_ha  = NULL
#       physical_rainfed_ha    = NULL
#       physical_split_available = False
# 3. Confirm source-total conservation from grid → country.
# 4. Update the country-level asset used by Notebook02:
#       01_SPAM_native_country_totals_QC.csv.gz
# 5. Do NOT modify any SPAM numerical values.
country_qc_parts = []
validation_rows = []

# 1. BUILD COUNTRY TOTALS FROM CURRENT NATIVE PARQUETS
for year in OBSERVED_YEARS:

    path = (
        OUTPUT_DIR
        / f"01_SPAM_native_5arcmin_{year}.parquet"
    )

    if not path.exists():

        raise FileNotFoundError(
            path
        )

    print(
        "\n" + "=" * 80
    )

    print(
        f"COUNTRY QC — SPAM {year}"
    )

    print(
        "=" * 80
    )

    # Duplicate native keys
    duplicate_count = duckdb.sql(
        f"""
        SELECT

            COUNT(*)
            -
            COUNT(
                DISTINCT
                grid_id_5min
                || '|'
                || country_iso3
                || '|'
                || CAST(
                    year AS VARCHAR
                )
                || '|'
                || source_product_code
            )

        FROM read_parquet(
            '{path.as_posix()}'
        )
        """
    ).fetchone()[0]

    # Negative-area check
    # NULL OTHE IR/RF values are valid and do not count as negative.
    negative_count = duckdb.sql(
        f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{path.as_posix()}'
        )

        WHERE

            harvested_irrigated_ha < 0

            OR harvested_rainfed_ha < 0

            OR harvested_total_ha < 0

            OR physical_irrigated_ha < 0

            OR physical_rainfed_ha < 0

            OR physical_total_ha < 0
        """
    ).fetchone()[0]

    if duplicate_count != 0:

        raise RuntimeError(
            f"Duplicate native 5′ keys found in {year}: "
            f"{duplicate_count}"
        )

    if negative_count != 0:

        raise RuntimeError(
            f"Negative final native areas found in {year}: "
            f"{negative_count}"
        )

    # 2000
    # Only the repaired 2000 parquet contains the explicit
    # physical_total_only_available flag.
    if year == 2000:

        country = duckdb.sql(
            f"""
            SELECT

                country_iso3,

                year,

                source_product_code,

                SUM(
                    harvested_irrigated_ha
                )
                    AS harvested_irrigated_ha,

                SUM(
                    harvested_rainfed_ha
                )
                    AS harvested_rainfed_ha,

                SUM(
                    harvested_total_ha
                )
                    AS harvested_total_ha,

                SUM(
                    harvested_total_source_ha
                )
                    AS harvested_total_source_ha,

                SUM(
                    physical_irrigated_ha
                )
                    AS physical_irrigated_ha,

                SUM(
                    physical_rainfed_ha
                )
                    AS physical_rainfed_ha,

                SUM(
                    physical_total_ha
                )
                    AS physical_total_ha,

                SUM(
                    physical_total_source_ha
                )
                    AS physical_total_source_ha,

                BOOL_OR(
                    physical_total_only_available
                )
                    AS physical_total_only_available,

                CASE

                    WHEN BOOL_OR(
                        physical_total_only_available
                    )

                    THEN FALSE

                    ELSE TRUE

                END
                    AS physical_split_available,

                CASE

                    WHEN BOOL_OR(
                        physical_total_only_available
                    )

                    THEN
                        'SOURCE_TOTAL_ONLY_IR_RF_SPLIT_UNAVAILABLE'

                    ELSE
                        'SOURCE_COMPONENTS_AVAILABLE'

                END
                    AS physical_component_status

            FROM read_parquet(
                '{path.as_posix()}'
            )

            GROUP BY

                country_iso3,

                year,

                source_product_code
            """
        ).df()

    # 2005 / 2010 / 2020
    else:

        country = duckdb.sql(
            f"""
            SELECT

                country_iso3,

                year,

                source_product_code,

                SUM(
                    harvested_irrigated_ha
                )
                    AS harvested_irrigated_ha,

                SUM(
                    harvested_rainfed_ha
                )
                    AS harvested_rainfed_ha,

                SUM(
                    harvested_total_ha
                )
                    AS harvested_total_ha,

                SUM(
                    harvested_total_source_ha
                )
                    AS harvested_total_source_ha,

                SUM(
                    physical_irrigated_ha
                )
                    AS physical_irrigated_ha,

                SUM(
                    physical_rainfed_ha
                )
                    AS physical_rainfed_ha,

                SUM(
                    physical_total_ha
                )
                    AS physical_total_ha,

                SUM(
                    physical_total_source_ha
                )
                    AS physical_total_source_ha,

                FALSE
                    AS physical_total_only_available,

                TRUE
                    AS physical_split_available,

                'SOURCE_COMPONENTS_AVAILABLE'
                    AS physical_component_status

            FROM read_parquet(
                '{path.as_posix()}'
            )

            GROUP BY

                country_iso3,

                year,

                source_product_code
            """
        ).df()

    country_qc_parts.append(
        country
    )

    validation_rows.append(
        {
            "year":
                year,

            "duplicate_native_keys":
                int(
                    duplicate_count
                ),

            "negative_rows":
                int(
                    negative_count
                ),

            "countries":
                int(
                    country[
                        "country_iso3"
                    ].nunique()
                ),

            "source_crops":
                int(
                    country[
                        "source_product_code"
                    ].nunique()
                ),

            "status":
                "PASS",
        }
    )

# 2. COMBINE COUNTRY TABLES
COUNTRY_TOTALS_QC = pd.concat(
    country_qc_parts,
    ignore_index=True,
)

# 3. COUNTRY-LEVEL HARVESTING INTENSITIES
COUNTRY_TOTALS_QC[
    "harvesting_intensity_irrigated"
] = safe_divide(
    COUNTRY_TOTALS_QC[
        "harvested_irrigated_ha"
    ],
    COUNTRY_TOTALS_QC[
        "physical_irrigated_ha"
    ],
)

COUNTRY_TOTALS_QC[
    "harvesting_intensity_rainfed"
] = safe_divide(
    COUNTRY_TOTALS_QC[
        "harvested_rainfed_ha"
    ],
    COUNTRY_TOTALS_QC[
        "physical_rainfed_ha"
    ],
)

COUNTRY_TOTALS_QC[
    "harvesting_intensity_total"
] = safe_divide(
    COUNTRY_TOTALS_QC[
        "harvested_total_ha"
    ],
    COUNTRY_TOTALS_QC[
        "physical_total_ha"
    ],
)

# OTHE total intensity is not interpretable because harvested OTHE and
# physical OTHE do not share a usable IR/RF source structure.

othe_total_only_mask = (
    COUNTRY_TOTALS_QC[
        "physical_total_only_available"
    ]
    .fillna(
        False
    )
)

COUNTRY_TOTALS_QC.loc[
    othe_total_only_mask,
    [
        "harvesting_intensity_irrigated",
        "harvesting_intensity_rainfed",
        "harvesting_intensity_total",
    ],
] = np.nan

# 4. SPAM2000 OTHE COUNTRY-LEVEL QC
OTHE_COUNTRY_QC = (
    COUNTRY_TOTALS_QC.loc[
        COUNTRY_TOTALS_QC[
            "year"
        ].eq(
            2000
        )
        &
        COUNTRY_TOTALS_QC[
            "source_product_code"
        ].astype(
            "string"
        )
        .str.lower()
        .eq(
            "othe"
        )
    ]
    .copy()
)

print(
    "\n=== SPAM2000 OTHE COUNTRY-LEVEL QC ==="
)

print(
    "Countries:",
    OTHE_COUNTRY_QC[
        "country_iso3"
    ].nunique(),
)

print(
    "OTHE physical total (ha):",
    f"{OTHE_COUNTRY_QC['physical_total_ha'].sum():,.3f}",
)

print(
    "OTHE published source physical total (ha):",
    f"{OTHE_COUNTRY_QC['physical_total_source_ha'].sum():,.3f}",
)

print(
    "Countries with IR/RF split unavailable:",
    int(
        (
            ~OTHE_COUNTRY_QC[
                "physical_split_available"
            ]
        ).sum()
    ),
)

print(
    "Countries with non-null physical IR:",
    int(
        OTHE_COUNTRY_QC[
            "physical_irrigated_ha"
        ].notna()
        .sum()
    ),
)

print(
    "Countries with non-null physical RF:",
    int(
        OTHE_COUNTRY_QC[
            "physical_rainfed_ha"
        ].notna()
        .sum()
    ),
)

display(
    OTHE_COUNTRY_QC.head(
        20
    )
)

# 5. GRID → COUNTRY OTHE TOTAL CONSERVATION
native_2000_path = (
    OUTPUT_DIR
    / "01_SPAM_native_5arcmin_2000.parquet"
)

GRID_OTHE_TOTAL = duckdb.sql(
    f"""
    SELECT

        SUM(
            physical_total_ha
        )

    FROM read_parquet(
        '{native_2000_path.as_posix()}'
    )

    WHERE lower(
        trim(
            source_product_code
        )
    ) = 'othe'
    """
).fetchone()[0]

COUNTRY_OTHE_TOTAL = (
    OTHE_COUNTRY_QC[
        "physical_total_ha"
    ]
    .sum()
)

OTHE_GRID_TO_COUNTRY_PASS = bool(
    np.isclose(
        GRID_OTHE_TOTAL,
        COUNTRY_OTHE_TOTAL,
        rtol=1e-10,
        atol=1e-3,
    )
)

OTHE_SOURCE_TO_COUNTRY_PASS = bool(
    np.isclose(
        OTHE_COUNTRY_QC[
            "physical_total_source_ha"
        ].sum(),
        COUNTRY_OTHE_TOTAL,
        rtol=1e-10,
        atol=1e-3,
    )
)

OTHE_COUNTRY_SPLIT_PASS = bool(
    OTHE_COUNTRY_QC[
        "physical_irrigated_ha"
    ]
    .isna()
    .all()

    and

    OTHE_COUNTRY_QC[
        "physical_rainfed_ha"
    ]
    .isna()
    .all()

    and

    (
        ~OTHE_COUNTRY_QC[
            "physical_split_available"
        ]
    )
    .all()
)

OTHE_COUNTRY_COUNT_PASS = bool(
    OTHE_COUNTRY_QC[
        "country_iso3"
    ]
    .nunique()
    ==
    176
)

# 6. SAVE UPDATED COUNTRY ASSET USED BY NOTEBOOK02
country_output_file = (
    OUTPUT_DIR
    / "01_SPAM_native_country_totals_QC.csv.gz"
)

COUNTRY_TOTALS_QC.to_csv(
    country_output_file,
    index=False,
    compression="gzip",
)

# 7. UPDATE MAIN RAW PROCESSING QC FOR 2000
# Preserve existing 2005 / 2010 / 2020 QC rows.
# Replace only the 2000 H/A rows with the repaired results.
main_processing_qc_file = (
    QC_DIR
    / "01_raw_processing_QC.csv"
)

if main_processing_qc_file.exists():

    OLD_PROCESSING_QC = pd.read_csv(
        main_processing_qc_file,
        low_memory=False,
    )

    OLD_PROCESSING_QC[
        "year"
    ] = pd.to_numeric(
        OLD_PROCESSING_QC[
            "year"
        ],
        errors="coerce",
    )

    OLD_PROCESSING_QC = (
        OLD_PROCESSING_QC.loc[
            ~OLD_PROCESSING_QC[
                "year"
            ].eq(
                2000
            )
        ]
        .copy()
    )

    UPDATED_PROCESSING_QC = pd.concat(
        [
            OLD_PROCESSING_QC,
            PROCESSING_QC_2000_REPAIR,
        ],
        ignore_index=True,
        sort=False,
    )

else:

    UPDATED_PROCESSING_QC = (
        PROCESSING_QC_2000_REPAIR.copy()
    )

UPDATED_PROCESSING_QC = (
    UPDATED_PROCESSING_QC
    .sort_values(
        [
            "year",
            "area_code",
        ]
    )
    .reset_index(
        drop=True
    )
)

UPDATED_PROCESSING_QC.to_csv(
    main_processing_qc_file,
    index=False,
)

# 8. FINAL VALIDATION TABLE
VALIDATION = pd.DataFrame(
    validation_rows
)

VALIDATION.to_csv(
    QC_DIR
    / "01_final_validation.csv",
    index=False,
)

display(
    VALIDATION
)

# 9. REPAIR STATUS
CELL7_PASS = bool(

    VALIDATION[
        "duplicate_native_keys"
    ]
    .eq(
        0
    )
    .all()

    and

    VALIDATION[
        "negative_rows"
    ]
    .eq(
        0
    )
    .all()

    and

    OTHE_GRID_TO_COUNTRY_PASS

    and

    OTHE_SOURCE_TO_COUNTRY_PASS

    and

    OTHE_COUNTRY_SPLIT_PASS

    and

    OTHE_COUNTRY_COUNT_PASS
)

CELL7_REPAIR_STATUS = pd.DataFrame(
    [
        {
            "SPAM2000_OTHE_grid_physical_total_ha":
                GRID_OTHE_TOTAL,

            "SPAM2000_OTHE_country_physical_total_ha":
                COUNTRY_OTHE_TOTAL,

            "OTHE_grid_to_country_conservation":
                OTHE_GRID_TO_COUNTRY_PASS,

            "OTHE_source_to_country_conservation":
                OTHE_SOURCE_TO_COUNTRY_PASS,

            "OTHE_IR_RF_split_remains_unresolved":
                OTHE_COUNTRY_SPLIT_PASS,

            "OTHE_country_count_176":
                OTHE_COUNTRY_COUNT_PASS,

            "country_QC_asset_updated":
                country_output_file.exists(),

            "status":
                (
                    "PASS_NOTEBOOK01_OTHE_REPAIR_FINALIZED"
                    if CELL7_PASS
                    else
                    "REVIEW_REQUIRED"
                ),
        }
    ]
)

print(
    "\n=== CELL 7 REPAIR STATUS ==="
)

display(
    CELL7_REPAIR_STATUS
)

CELL7_REPAIR_STATUS.to_csv(
    QC_DIR
    / "01_REPAIR_2000_OTHE_country_status.csv",
    index=False,
)

# 10. STOP IF FAILED
if not CELL7_PASS:

    raise RuntimeError(
        "Notebook01 OTHE country-level repair did not pass. "
        "Do not continue to Notebook02."
    )

print(
    "\nNOTEBOOK 01 STATUS: "
    "PASS_NOTEBOOK01_OTHE_REPAIR_FINALIZED"
)

print(
    "\nUpdated Notebook02 input:"
)

print(
    country_output_file
)

print(
    "\nNext step: return to Notebook02. "
    "Do not use the previous OTHE=0 parent-audit result."
)

## Interpretation and next production step

This notebook preserves the legacy processing algorithms, including the **SPAM2000 OTHE total-only physical-area repair**. It is intentionally labeled a *review candidate* until the first-pass native SPAM production runner is integrated and a clean-run test demonstrates the same results from raw SPAM inputs.

**Before a public GitHub/Zenodo release:** (1) integrate initial production of all four native Parquet years, (2) re-run the OTHE rule within that initial production instead of relying on a repair backup, (3) verify source/version hashes, (4) confirm expected QC and output schema for downstream Notebook 02, and (5) record the clean-run results and the software environment. Scientific source claims should be supported by explicit references in the final methodology documentation.